# Stage 01: Diarization

Đảm bảo bạn đã add Dataset audio gốc (nếu là Stage 01), hoặc Output của Stage trước vào Kaggle Dataset.

In [ ]:
# Clone project từ nhánh test-divide-stage
import os
if not os.path.exists('/kaggle/working/sommelier'):
    !git clone -b test-divide-stage https://github.com/lamkdhe180931-arch/sommelier.git
else:
    !cd /kaggle/working/sommelier && git pull origin test-divide-stage
%cd /kaggle/working/sommelier/podcast-pipeline/stages

In [ ]:
# Đăng nhập HuggingFace (Cần thiết cho Pyannote)
# BẠN CẦN TẠO SECRET CÓ TÊN LÀ HF_TOKEN TRONG KAGGLE SECRETS TRƯỚC NHÉ
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    import json
    if os.path.exists('config.json'):
        with open('config.json', 'r') as f: cfg = json.load(f)
        cfg['huggingface_token'] = hf_token
        with open('config.json', 'w') as f: json.dump(cfg, f, indent=4)
    print('Đã load HF_TOKEN thành công!')
except Exception as e:
    print('Chưa cấu hình HF_TOKEN trong Kaggle Secrets. Nếu chạy lỗi, vui lòng cấu hình HF_TOKEN.')


In [ ]:
!pip install -q nemo_toolkit[asr] pyannote.audio
!pip install -q soundfile librosa pandas pydub onnxruntime-gpu

In [ ]:
AUDIO_INPUT = '/kaggle/input/your-dataset/audio.wav' # THAY ĐỔI ĐƯỜNG DẪN NÀY
OUT_JSON = '/kaggle/working/diarization.json'

# ==========================================
# THAM SỐ TINH CHỈNH MODEL
# ==========================================
SORTFORMER_MODEL_NAME = 'nvidia/diar_streaming_sortformer_4spk-v2.1'

# 1. Model Sortformer (Thuật toán Binarize & Padding)
ONSET = 0.53         # Ngưỡng bắt đầu giọng nói (Tăng -> Cắt gắt hơn)
OFFSET = 0.49        # Ngưỡng kết thúc giọng nói (Giảm -> Kéo dài đuôi câu hơn)
MIN_DUR_ON = 0.42    # Đoạn nói tối thiểu (giây). Ngắn hơn mức này bị xoá bỏ
MIN_DUR_OFF = 0.34   # Khoảng lặng tối thiểu (giây). Ngắn hơn mức này bị gộp làm một
PAD_ONSET = 0.23     # Kéo mốc thời gian bắt đầu ra trước (giây) tránh mất chữ cái đầu
PAD_OFFSET = 0.01    # Kéo mốc thời gian kết thúc ra sau (giây)

# 2. Model Pyannote (Nối Speaker các Chunk)
SPK_LINK_TH = 0.6          # Ngưỡng giống nhau để nối người nói giữa các đoạn âm thanh. (Tăng -> Khó gộp hơn)
SPK_RECLUSTER_TH = 0.75    # Ngưỡng dọn dẹp ID vụn ở bước cuối cùng. (Tăng -> Khó gộp nhóm hơn)

!python stage_01_diarize.py \
  --input_audio "{AUDIO_INPUT}" \
  --out "{OUT_JSON}" \
  --sortformer-model-name "{SORTFORMER_MODEL_NAME}" \
  --onset {ONSET} \
  --offset {OFFSET} \
  --min-duration-on {MIN_DUR_ON} \
  --min-duration-off {MIN_DUR_OFF} \
  --sortformer-pad-onset {PAD_ONSET} \
  --sortformer-pad-offset {PAD_OFFSET} \
  --speaker-link-threshold {SPK_LINK_TH} \
  --speaker-recluster-threshold {SPK_RECLUSTER_TH}

In [ ]:
# ==========================================
# CELL ĐÁNH GIÁ NHANH KẾT QUẢ DIARIZATION
# ==========================================
import json
import librosa
import IPython.display as ipd
from IPython.core.display import display, HTML

try:
    with open(OUT_JSON, 'r') as f:
        data = json.load(f)
    segments = data.get('segments', [])
    if not segments:
        print('Không tìm thấy phân đoạn nào trong JSON!')
    else:
        print(f'Tổng cộng {len(segments)} đoạn. Đang tải audio để hiển thị 30 đoạn đầu tiên...')
        waveform, sr = librosa.load(AUDIO_INPUT, sr=16000)
        html_out = "<table border='1' style='width:100%; text-align:center;'>"
        html_out += "<tr><th>Speaker ID</th><th>Thời gian</th><th>Nghe thử</th></tr>"
        
        for seg in segments[:30]:
            spk = seg['speaker']
            start, end = seg['start'], seg['end']
            start_sample = int(start * sr)
            end_sample = int(end * sr)
            clip = waveform[start_sample:end_sample]
            
            # Tạo thẻ <audio> bằng IPython.display
            audio_widget = ipd.Audio(data=clip, rate=sr)
            audio_html = audio_widget._repr_html_()
            
            html_out += f"<tr><td><b>{spk}</b></td><td>{start:.2f}s - {end:.2f}s</td><td>{audio_html}</td></tr>"
        html_out += "</table>"
        display(HTML(html_out))
except Exception as e:
    print('Có lỗi khi tạo bảng nghe thử:', e)


In [ ]:
import os
import subprocess
from IPython.display import FileLink

subprocess.run(['zip', '-r', '/kaggle/working/diarization_output.zip', '/kaggle/working/diarization.json'])
FileLink('/kaggle/working/diarization_output.zip')